Install packages

In [2]:
!pip install transformers datasets peft accelerate bitsandbytes trl -q
!pip install unsloth -q

import transformers
import peft

Load Model

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Phi-3-mini-4k-instruct",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)
tokenizer.pad_token = tokenizer.eos_token

print('Model loaded successfully.')
print(f'Model type: {type(model)}')

Apply LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj',
                      'up_proj', 'down_proj'],
    lora_alpha = 16,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = True,
    random_state = 42
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Trainable % : {100 * trainable / total:.2f}%")

Prepare Dataset

In [ ]:
import sys
sys.path.append(PROJECT_DIR)
exec(open(f'{PROJECT_DIR}/src/data/chemistry_dataset.py').read())

from datasets import Dataset

def format_sample(sample):
    return {
        "text": f"""### Instruction:\n{sample['instruction']}\n\n### Response:\n{sample['response']}{tokenizer.eos_token}"""
    }

cleaned = []
for s in chemistry_qa:
    if len(s['response']) > 20 and s['response'].endswith('.'):
        cleaned.append(s)

formatted = [format_sample(s) for s in cleaned]

dataset   = Dataset.from_list(formatted)

print(f"Total samples : {len(dataset)}")
print(f"\nSample entry:")
print(dataset[0]['text'][:300])


Fine-tune with SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,

    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,

        num_train_epochs = 3,
        learning_rate = 1e-4,

        fp16 = True,
        logging_steps = 10,

        save_strategy = "epoch",
        output_dir = f'{PROJECT_DIR}/models/llm_checkpoints',

        report_to = "none",
    )
)

print("Starting fine-tuning...")
trainer.train()
print("Fine-tuning complete.")

Save model

In [ ]:
# Save LoRA adapters to Drive
SAVE_PATH = f'{PROJECT_DIR}/models/llm_finetuned'
os.makedirs(SAVE_PATH, exist_ok=True)

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"Model saved to: {SAVE_PATH}")
!ls -lh {SAVE_PATH}

Test the fine-tuned model

In [ ]:
FastLanguageModel.for_inference(model)

def ask_chemistry(question: str) -> str:
    prompt = f"""### Instruction:
Answer concisely and factually. Do not explain.

Question: {question}

### Response:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
    **inputs,
    max_new_tokens=40,
    max_length=None,
    temperature=0.0,
    top_p=1.0,
    repetition_penalty=1.2,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
)

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("### Response:")[-1].strip()

# Test questions
questions = [
    "What is the HOMO-LUMO gap?",
    "What does a small HOMO-LUMO gap indicate?",
    "What is the QM9 dataset?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask_chemistry(q)}")
    print("-" * 60)